# Minimal demo

In [ ]:
%load_ext autoreload
%autoreload 2

import mne
from pathlib import Path
from mne.preprocessing import find_eog_events

real_data_dir = Path("data/real_data")
edf_path_demo = real_data_dir / "test_from_20s.edf"
raw_demo = mne.io.read_raw_edf(edf_path_demo, preload=True, verbose=False)
raw_demo.drop_channels(['ECG  ECG'])

events_demo = find_eog_events(raw_demo, ch_name='EEG FP1-A1')

In [ ]:
import matplotlib
matplotlib.use('TkAgg')

raw_demo.plot(
    events=events_demo,
    duration=30,  # Show 30 seconds initially
    start=0,
    n_channels=15,
    scalings='auto',
    show=True,
)

# Test processing

In [ ]:
import numpy as np

def events_to_times(raw: mne.io.BaseRaw, events: np.ndarray) -> np.ndarray:
    return (events[:, 0] - raw.first_samp) / raw.info['sfreq']

def save_txt(times: np.ndarray, out_path: Path) -> None:
    np.savetxt(str(out_path), times, fmt="%.4f")

times_demo = events_to_times(raw_demo, events_demo)
save_txt(times_demo, edf_path_demo.parent / "fp1_eyem.txt")

In [ ]:
times_demo

In [ ]:
def consensus(times1: np.ndarray, times2: np.ndarray, delta_sec: float) -> np.ndarray:
    """
    Combine two sorted arrays by averaging values
    that are within `delta_sec` of each other.
    """
    result = []
    used_in_list2 = set()
    
    for val1 in times1:
        closest_dist = float('inf')
        closest_idx = -1
        closest_val = None
        
        for idx, val2 in enumerate(times2):
            if idx in used_in_list2:
                continue
                
            dist = np.abs(val1 - val2)
            if dist < delta_sec and dist < closest_dist:
                closest_dist = dist
                closest_idx = idx
                closest_val = val2
        
        if closest_idx != -1:
            result.append((val1 + closest_val) / 2)
            used_in_list2.add(closest_idx)
    
    return np.array(sorted(result))


def process(edf_path: Path, out_dir: Path | None = None, delta_sec: float = 0.05) -> None:
    """
    1. finds all eog events in FP1 and FP2 channels, saves to TXT
    2. merges two events list using `delta_sec` as max diff
    """
    if out_dir is None:
        out_dir = edf_path.parent
    
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    chans = ["FP1-A1", "FP2-A2"]
    chans_to_times = {}
    for ch in chans:
        events = find_eog_events(raw, ch_name=f"EEG {ch}", verbose=False)
        times = events_to_times(raw, events)
        chans_to_times[ch] = times
        save_txt(times, out_dir / f"{edf_path.stem}_eog_{ch}.txt")
    
    consensus_times = consensus(chans_to_times[chans[0]], chans_to_times[chans[1]], delta_sec)
    save_txt(consensus_times, out_dir / f"{edf_path.stem}_eog_consensus.txt")


process(edf_path_demo)


# Real processing

In [ ]:
big_edf_paths = [
    "/mnt/y/vzuev/datasets/eeg_videos/vkuklin_july19_2025/vkuklin_july19_2025 18-JUL-2025_21h28m18.382s.edf",
    "/mnt/y/vzuev/datasets/eeg_videos/vzuev_june18_2025/vzuev_june18_2025 17-JUN-2025_22h05m27.475s.edf",
]

In [ ]:
for edf_path_str in big_edf_paths:
    big_edf_p = Path(edf_path_str)
    process(big_edf_p, real_data_dir / "big_edfs")
